1. Activation Functions 
2. Optimizers 
3. Attention Mechanism

**Transformer Attention Mechanism**

In [65]:
import numpy as np

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def self_attention(Q,K,V):
    """
    attention =  softmax((Q.KT/sqrt(K)).V
    """
    scores = softmax((Q @ K.T) / np.sqrt(K.shape[-1]))@V
    return scores
    


In [67]:
Q = np.array([[1, 0], [0, 1]])
K = np.array([[1, 0], [0, 1]])
V = np.array([[1, 2], [3, 4]])

output = self_attention(Q, K, V)
output

array([[1.6604769, 2.6604769],
       [2.3395231, 3.3395231]])

**MultiHead Attention**

In [42]:
import numpy as np

def compute_qkv(X, W_q, W_k, W_v):
    """Computes Query, Key, and Value matrices."""
    Q = X @ W_q
    K = X @ W_k
    V = X @ W_v
    return Q, K, V

def self_attention(Q, K, V):
    """
    Computes scaled dot-product attention for all heads simultaneously.
    Expects input shapes: (n_heads, seq_len, d_k)
    """
    d_k = Q.shape[-1]
    
    # 1. Batched dot-product: (n_heads, seq_len, d_k) @ (n_heads, d_k, seq_len)
    # We use np.swapaxes to transpose only the last two dimensions of K
    scores = (Q @ np.swapaxes(K, -1, -2)) / np.sqrt(d_k)
    
    # 2. Numerically stable softmax along the last axis (seq_len)
    scores_max = np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # 3. Multiply by Value matrix: (n_heads, seq_len, seq_len) @ (n_heads, seq_len, d_k)
    output = attention_weights @ V
    
    return output

def multi_head_attention(Q, K, V, n_heads):
    """
    Vectorized multi-head attention without loops.
    """
    seq_len, d_model = Q.shape
    d_k = d_model // n_heads
    
    # 1. Reshape to split into heads: (seq_len, n_heads, d_k)
    # 2. Transpose to process heads as a batch: (n_heads, seq_len, d_k)
    Q_split = Q.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    K_split = K.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    V_split = V.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    
    # 3. Compute attention for all heads at once
    # Resulting shape: (n_heads, seq_len, d_k)
    head_outputs = self_attention(Q_split, K_split, V_split)
    
    # 4. Revert the transpose: (seq_len, n_heads, d_k)
    # 5. Reshape (flatten the last two dimensions) back to: (seq_len, d_model)
    final_output = head_outputs.transpose(1, 0, 2).reshape(seq_len, d_model)
    
    return final_output

# --- Test Case ---

X = np.array([[1, 2, 3, 4], 
                [5, 6, 7, 8]])  # (2, 4)

W_q = W_k = W_v = np.eye(4)

Q, K, V = compute_qkv(X, W_q, W_k, W_v)
result = multi_head_attention(Q, K, V, n_heads=2)

print("Vectorized Output Matrix:\n", result)

Vectorized Output Matrix:
 [[4.99917423 5.99917423 6.99999999 7.99999999]
 [5.         6.         7.         8.        ]]


**Masked Self Attention**